# 模块&包的导入和运行

## 标准库与模块导入
- 模块是组织代码的一种单位，普通的 .py 文件可以作为模块
- 标准库是随 Python 提供的模块集合

### 对象、命名空间、作用域
- 对象是 Python 运行时的实体，具有身份、类型和值或状态
- 命名空间保存“名字 → 对象”的对应关系（变量）
- 作用域决定名字的查找路径，决定执行某一处代码时，可以到哪些命名空间里查找名字

### 加载视角的模块导入

- 模块导入可以抽象为几个层次：

| 层次 | 含义 | 例子 |
|---|---|---|
| 模块的实现来源 | 模块内容实际存放或实现的位置 | `statistics.py`、`.pyd`、内置模块 |
| 运行时模块对象 | Python 运行时用于承载模块内容的对象 | `statistics` 模块对象 |
| 当前作用域中的名字 | 当前代码中用来引用模块对象的名字 | `statistics`、`st`、其他写定的别名 |

- `import` 的整体流程可以抽象为：查找模块 -> 取得模块对象并登记 -> 初始化模块，绑定名字到主脚本命名空间
    1. 查找模块的实现来源
    2. 创建或获得模块对象
    3. 将模块对象登记到 `sys.modules`
    4. 执行该模块对应的初始化过程
    5. 在当前作用域中绑定相应名字

- 不同类型模块的“初始化过程”可能不同：
    - 纯 Python 模块：执行 `.py` 文件的顶层代码
        - 顶层代码指直接写在模块最外层的代码
        - 函数体里的代码不属于顶层代码，调用时才执行
    - 普通 Python 包：通常执行包中的 `__init__.py`
    - C 扩展模块：执行扩展模块自身的初始化逻辑
    - 内置模块：由 Python 解释器内部完成初始化

- 对于 `statistics`：
    - 找到 `statistics.py`
    - 创建 `statistics` 模块对象并准备模块命名空间
    - 将模块对象登记到 `sys.modules`
    - 在该模块自己的命名空间中执行 `statistics.py` 的顶层代码
        - 文件中的函数、类、变量等因此进入模块命名空间
    - `import statistics` 最后在当前作用域中绑定名字 `statistics`，使其指向该模块对象

- 如果模块已经存在于 `sys.modules` 中，通常会直接复用已有模块对象，而不会再次完整执行初始化过程

### 命名空间视角下的模块导入

- 每个模块都有自己的全局命名空间，彼此独立

- 模块中的函数、类、变量，本质上都是模块命名空间中的名字绑定

- `import statistics`

  - 在当前命名空间中绑定名字 `statistics`
  - `statistics` 指向模块对象
  - 再通过属性访问找到模块中的对象
  - `statistics.mean `取得函数对象，没有调用
  - `statistics.mean([10, 20, 30])`取得函数对象，再调用，得到返回值

- `from statistics import mean`

  - 直接取得 `statistics` 模块中的 `mean` 函数对象
  - 在当前命名空间中绑定名字 `mean`
  - 当前命名空间中的`mean` 和 `statistics.mean` 最初指向同一个函数对象
  - 不是复制对象，只是多了一个名字绑定
  - 如果使 `mean = 100` 
    - 当前命名空间：mean → 整数 100
    - 模块命名空间：statistics.mean → 原函数对象

- `as` 只是改变当前命名空间中使用的名字

  - `import statistics as st`
  - `from statistics import mean as avg`

- 核心区别

  - `import statistics`：绑定模块对象
  - `from statistics import mean`：直接绑定模块中的对象


## `__name__` 与入口保护

### `__name__`定义
-` __name__` 用来标识当前模块在 Python 导入系统里的名字，通常由 Python 在执行模块代码之前设置，绑定到一个字符串
- 每个模块有自己的 `__name__`，彼此独立
- `__name__`的值由模块的运行方式决定，可以理解为"角色名"
    - 作为主模块文件执行时，其值为 `__main__`
    - 通过普通导入加载时，`__name__` 通常等于该模块的完整导入名


### 入口保护
- 用条件判断把“作为程序入口时才应该运行的代码”保护起来，避免模块被 import 时意外执行
- 通过`if __name__ == '__main__'`判断当前文件是否作为主模块运行
- 在这个条件的缩进里写特殊的处理代码，使其只在作为主模块运行时才执行

### 其他用途
1. 调试和日志里标识当前模块
2. 判断当前代码属于哪个模块
...


## 拓展到包的模块导入

### 模块与包

- `import` 的核心仍然是取得并绑定模块对象
- 包也属于模块体系，只是可以继续包含子模块和子包
  - 普通 `.py` 文件 → 普通模块
  - 包目录 → 包模块
- 因此复杂包只是模块结构的递归嵌套
  - openpyxl(包模块对象)
    - styles（子包模块对象）
      - fonts.py（子模块对象）
        - class Font(...)（模块内部对象）
        - ...（模块内部对象）
    - workbook（子包模块对象）
      - ...（子模块对象）
    - ...（子包模块对象）

### 包的 `__init__.py`

- 普通模块初始化时执行 `.py` 文件顶层代码
- 普通包初始化时通常执行包目录中的 `__init__.py`
- `__init__.py` 相当于包这一层自己的初始化代码

  - 可以定义包级名字
  - 可以从内部模块导入对象，对外暴露到当前包的命名空间


- 以`openpyxl`包对象的`styles`子包对象的`fonts.py`子模块对象为例
- 如果 `Font` 定义在 `fonts.py`，而 `styles/__init__.py`：
    - `from .fonts import Font`
- 则可以：
    - `from openpyxl.styles import Font`

- 因此对象的 实际定义位置 和 对外导入位置 不一定相同

### 包中的导入

- `from openpyxl import styles`
  - 绑定 `openpyxl.styles` 子包模块对象
- `from openpyxl.styles import Font`
  - 绑定 `openpyxl.styles` 中的 `Font` 类对象
- `from A import B` 中的 `B` 可以是函数、类、变量，也可以是子模块或子包

### 包的 `__main__.py`

- 普通主脚本不需要命名为 `__main__.py`
- 包中的 `__main__.py` 用来定义“整个包作为程序启动时执行什么”

- mypackage
  - `__init__.py`
  - `__main__.py`

1. `import mypackage`:初始化包，通常执行 `__init__.py`
2. `python -m mypackage`:将包作为程序启动，执行 `__main__.py`

## 统一模型

-  普通模块和包不是两套机制,都遵循：
    1. 检查 `sys.modules`
        - 已存在，直接取得已有模块对象，在当前命名空间绑定名字
        - 不存在，继续
    2. 查找模块的实现来源
    3. 创建模块对象并建立模块命名空间
        - 在这里设置 `__name__`、`__package__`、`__spec__` 等属性
    4. 登记到 `sys.modules`
    5. 初始化模块
        - 纯 Python 模块：执行 `.py` 文件顶层代码
        - 普通包：通常执行 `__init__.py`
        - ...
    6. 在当前命名空间绑定相应名字

- 包只是在模块内部继续包含模块，使这一结构可以递归扩展